# Рейтинг о качестве жизни женщин

In [135]:
import pandas as pd
import openpyxl
import numpy as np

# Общие функции

In [136]:
# # database of unique regions
# obstetric['Название региона'].unique()

# # turn array into dataframe
# regions_df = pd.DataFrame(obstetric['Название региона'].unique(), columns=['Название региона'])
# regions_df = regions_df.sort_values(by='Название региона', ascending=True)
# regions_df.sort_values(by='Название региона', ascending=True).head()
# regions_df.to_clipboard()

In [137]:
# # сравниваю данные ЕБТ с данными Росстата по регионам
# region_keys = pd.read_csv(
#         '/Users/Helen/Downloads/problem_gender_inequality_all - названия регионов.csv'
# )

# region_keys.head()

# # using region_keys['ЕБТ'] assign region_keys['код региона КОУЖ'] to every region in ebt['Название региона']
# ebt = ebt.merge(region_keys, left_on='Название региона', right_on='ЕБТ', how='left')
# ebt.head()

# comparison = pd.read_csv(
#         '/Users/Helen/Downloads/problem_gender_inequality_all - Sheet4.csv'
# )
# comparison.head()

# ebt = ebt.merge(comparison, left_on='код региона КОУЖ', right_on='код региона КОУЖ', how='left')

In [138]:
# загружаю названия регионов с кодами ОКАТО 
#get spreadsheets key from url
gsheetkey = "1sUJkTaeDhOQdEzw5FhFqCsYvEeaLoqVz9oWxzCgiGDI"
# https://docs.google.com/spreadsheets/d/1sUJkTaeDhOQdEzw5FhFqCsYvEeaLoqVz9oWxzCgiGDI/edit?usp=sharing

#sheet name
sheet_name = 'список регионов sorted'

url=f'https://docs.google.com/spreadsheet/ccc?key={gsheetkey}&output=xlsx'
lookup = pd.read_excel(url,sheet_name=sheet_name)

In [139]:
# функция для присваивания ОКАТО регионам по их названиям
# lookup = pd.read_csv('/Users/Helen/Downloads/список регионов sorted.csv')

def vlookup_okato(df, lookup_df, region_col):
    mapping = (
        lookup_df.dropna(subset=[region_col])
        .drop_duplicates(subset=[region_col])
        .set_index(region_col)['object_okato']
        .astype(str)
    )
    # df['Номер региона (ОКАТО)'] = df['Название региона'].map(mapping)
    new_col = df['Название региона'].map(mapping)
    df.insert(loc = 0,
          column = 'Номер региона (ОКАТО)',
          value = new_col)
    return df

In [140]:
# присваиваю ОКАТО по кодам регионов КОУЖ
# lookup = pd.read_csv('/Users/Helen/Downloads/список регионов sorted.csv')

def vlookup_kouzh(df, lookup_df, region_col):
    mapping = (
        lookup_df.dropna(subset=[region_col])
        .drop_duplicates(subset=[region_col])
        .set_index(region_col)['object_okato']
        .astype(str)
    )
    new_col = df['Номер региона (КОУЖ)'].map(mapping)
    df.insert(loc = 0,
          column = 'Номер региона (ОКАТО)',
          value = new_col)
    return df

In [141]:
year_cols = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# Загрузка новых данных

### Средняя начисленная зарплата по регионам 2021, 2023 годы

In [142]:
# функция для выделения и подсчета данных о 
# средней начисленной зарплате по полу
def extract_salary(year):
    # загружаю и обрабатываю данные
    salary = pd.read_excel(f'/Users/Helen/Downloads/sved_57-t_{year}/Статбюлетень {year}.xlsx', sheet_name='42')
    salary = salary.rename(columns={
                        # 'Unnamed: 1': 'Средняя заработная плата по группам занятий, все работники',
                        'Unnamed: 2': 'Средняя заработная плата по группам занятий, мужчины',
                        'Unnamed: 3': 'Средняя заработная плата по группам занятий, женщины',
                        f'42. Средняя начисленная заработная плата работников списочного состава по полу и \nсубъектам Российской Федерации за октябрь {year} года ': 'Название региона'
                        })
    salary['Название региона'] = salary['Название региона'].str.strip()
    salary['Название региона'] = salary['Название региона'].str.replace('\n', ' ')
    salary['Название региона'] = salary['Название региона'].str.replace('H', 'Н') # replace Latin H in Hижний Новгород with Cyrillic Н
    salary = salary.drop(columns=['Unnamed: 1', 'Unnamed: 4'])
    salary = salary.drop(index=[0, 1, 2])
    # salary.head()

    # считаю гендерный разрыв в оплате труда, как у «Если быть точным» (ЕБТ)
    salary['Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)'] = ((salary['Средняя заработная плата по группам занятий, мужчины'] - 
                                                                                    salary['Средняя заработная плата по группам занятий, женщины'])/
                                                                                    salary['Средняя заработная плата по группам занятий, мужчины']*100)

    # подгоняю данные под формат датасета ЕБТ
    salary_long_21 = salary.melt(id_vars=['Название региона'], 
                            value_vars=['Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)', 
                                        # 'Средняя заработная плата по группам занятий, все работники', 
                                        'Средняя заработная плата по группам занятий, мужчины', 
                                        'Средняя заработная плата по группам занятий, женщины'], 
                            var_name='Название показателя', value_name=str(year))

    salary_long_21['Номер показателя (ID)'] = salary_long_21['Название показателя'].map({
                                                    'Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)': 95799, 
                                                    'Средняя заработная плата по группам занятий, мужчины': 95789, 
                                                    # 'Средняя заработная плата по группам занятий, все работники': 95916, 
                                                    'Средняя заработная плата по группам занятий, женщины': 95790})

    salary_long_21 = vlookup_okato(salary_long_21, lookup, 'salary_long')
    salary_long_21.drop(salary_long_21[salary_long_21['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
    return salary_long_21

In [143]:
# подсчет данных по 2021 году
salary_long_21 = extract_salary(2021)
salary_long_21 = salary_long_21[['Номер региона (ОКАТО)', 
                                #  'Название региона', 
                                 'Номер показателя (ID)', 
                                 '2021']]

# подсчет данных по 2023 году
salary_long_23 = extract_salary(2023)
salary_long = salary_long_21.merge(
                                salary_long_23[['Номер региона (ОКАТО)', 'Номер показателя (ID)', '2023']], 
                                on=['Номер региона (ОКАТО)', 'Номер показателя (ID)'], 
                                how='left')
for j in [-2, -1]:    
    for i in range(len(salary_long)):
        if salary_long.iloc[i, j]:
            salary_long.iloc[i, j] = round(salary_long.iloc[i, j], 2)

salary_long.sort_values(by='Номер региона (ОКАТО)').head(20)

,Номер региона (ОКАТО),Номер показателя (ID),2021,2023
70,1000000,95799,21.84,26.31
158,1000000,95789,38463,56490
246,1000000,95790,30064,41628
171,10000000,95789,80460,109746
259,10000000,95790,51013,65700
83,10000000,95799,36.6,40.13
176,100000000000,95790,49205,64591
88,100000000000,95789,71364,96425
0,100000000000,95799,31.05,33.01
21,11000000,95799,36.53,32.66


### Уход за детьми, «Комплексное наблюдение условий жизни населения»

In [144]:
# функция для подсчета доли женщин, ухаживающих за детьми
# данные Комплексного наблюдения условий жизни населения (КОУЖ)
def calculate_share(group):
    answer = (group[group['I05_41'].isin([1, 2, 3, 4])]['KVZV'].sum()
                / group['KVZV'].sum()*100)
    return answer.round(0)

In [145]:
# функция для подсчета доли женщин, ухаживающих за детьми, в указанном году
def get_kouzh(year):
    kouzh = pd.read_csv(
        f'/Users/Helen/Downloads/КОУЖ (Комплексное наблюдение условий жизни населения)/{year}/КОУЖ_{year}_Датасет_по_индивидуальным_данным_основной.csv',
        sep=';')

    kouzh_zh = kouzh[kouzh['H01_01'] == 2] # выделить из общего датасета только женщин

    share_by_region = kouzh_zh.groupby('H00_02').apply(calculate_share)

    # make a dataset out of share_by_region
    share_by_region = share_by_region.reset_index()
    share_by_region = share_by_region.rename(columns={0: f'{year}',
                                                            'H00_02': 'Номер региона (КОУЖ)'})

    # добавим подсчет по всей стране
    answer = round(((kouzh_zh[kouzh_zh['I05_41'].isin([1, 2, 3, 4])]['KVZV'].sum()
                    / kouzh_zh['KVZV'].sum()*100)), 0)
    share_by_region = pd.concat([share_by_region, pd.DataFrame({'Номер региона (КОУЖ)': [100], f'{year}': [answer]})], ignore_index=True)
    return share_by_region

In [146]:
# подсчитываю доли и объединяю данные
share_by_region_22 = get_kouzh(2022)
share_by_region_24 = get_kouzh(2024)

# доля женщин, ухаживающих за детьми, в 2022 и 2024 году, сводная таблица
kouzh_combined = pd.merge(share_by_region_22, share_by_region_24, on='Номер региона (КОУЖ)', how='outer')
kouzh_combined['Номер показателя (ID)'] = 95853
# kouzh_combined['Описание'] = 'Рассчет «Горизонтальной России» по данным КОУЖ. Доля респондентов, в круг ежедневных занятий которых входит уход за детьми, от всех респондентов. Источник - Росstat: Комплексное наблюдение условий жизни населения.'
vlookup_kouzh(kouzh_combined, lookup, 'kouzh_code')
kouzh_combined.drop(kouzh_combined[kouzh_combined['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
kouzh_combined = kouzh_combined[['Номер региона (ОКАТО)', 
                   'Номер показателя (ID)', 
                   '2022', '2024']]

print(f"kouzh_combined has {kouzh_combined['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")
kouzh_combined.head()

/var/folders/5r/7bhf5tqj0wb9bbk62kmr6wk80000gp/T/ipykernel_2691/254446953.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  share_by_region = kouzh_zh.groupby('H00_02').apply(calculate_share)


kouzh_combined has 0 missing values in 'Номер региона (ОКАТО)' column.


/var/folders/5r/7bhf5tqj0wb9bbk62kmr6wk80000gp/T/ipykernel_2691/254446953.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  share_by_region = kouzh_zh.groupby('H00_02').apply(calculate_share)


,Номер региона (ОКАТО),Номер показателя (ID),2022,2024
0,1000000,95853,34.0,30.0
1,3000000,95853,36.0,33.0
2,4000000,95853,37.0,30.0
3,5000000,95853,41.0,36.0
4,7000000,95853,44.0,41.0


# Обновление датасета «Если быть точным»

In [147]:
# загружаю оригинальные данные «Если быть точным»
gsheetkey = "1sUJkTaeDhOQdEzw5FhFqCsYvEeaLoqVz9oWxzCgiGDI"
# https://docs.google.com/spreadsheets/d/1sUJkTaeDhOQdEzw5FhFqCsYvEeaLoqVz9oWxzCgiGDI/edit?usp=sharing
sheet_name = 'problem_gender_inequality_all'
url=f'https://docs.google.com/spreadsheet/ccc?key={gsheetkey}&output=xlsx'
ebt = pd.read_excel(url,sheet_name=sheet_name)

In [148]:
# выделяю из датасета «Если быть точным» только нужные мне показатели
ebt_use = ebt[ebt['Номер показателя (ID)'].isin([95799, 95789, 95790, 95853])]

# добавляю обозначение группы (м или ж) в описание показателя
mask = ebt_use['Номер показателя (ID)'] == 95789
ebt_use.loc[mask, 'Название показателя'] = ebt_use.loc[mask, 'Название показателя'] + ', мужчины'
mask = ebt_use['Номер показателя (ID)'] == 95790
ebt_use.loc[mask, 'Название показателя'] = ebt_use.loc[mask, 'Название показателя'] + ', женщины'

# преобразую строки в вещественные числа
ebt_use.loc[ebt_use['Название показателя'] == 'Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)', [i for i in range(2015, 2021, 2)]] = ebt_use.loc[ebt_use['Название показателя'] == 'Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)', [i for i in range(2015, 2021, 2)]].apply(lambda col: col.str.replace(',', '.', regex=False).astype(float))

ebt_use = ebt_use.drop(columns=['Группа показателей', 'Сегмент', 'Показывать', ' '])
    
ebt_use = vlookup_okato(ebt_use, lookup, 'ebt_use')
ebt_use.drop(ebt_use[ebt_use['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
# print(f"ebt_use has {ebt_use['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")

ebt_use = ebt_use[['Номер региона (ОКАТО)',
                   'Название региона', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения',      
                   'Источник', 
                   'Описание',
                   2014, 2015, 2016, 2017, 2018,
                   2019, 2020, 2021, 2022, 2023, 2024]]
# Replace "н/д" with NaN in the year columns you want to fill
ebt_use = ebt_use.replace('н/д', np.nan)

/var/folders/5r/7bhf5tqj0wb9bbk62kmr6wk80000gp/T/ipykernel_2691/1096726661.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ebt_use = ebt_use.replace('н/д', np.nan)


In [149]:
# добавляю данные о зарплатах и уходе за детьми
ebt_use = ebt_use.merge(
    salary_long,
    on=['Номер региона (ОКАТО)', 'Номер показателя (ID)'],
    how='left',
)
ebt_use[2021] = ebt_use[2021].fillna(ebt_use['2021'])
ebt_use[2023] = ebt_use[2023].fillna(ebt_use['2023'])
ebt_use.drop(columns=['2021', '2023'], inplace=True)
# дополняю описание показателя ЕБТ о гендерном разрыве в зарплатах
mask = ebt_use['Номер показателя (ID)'] == 95799
ebt_use.loc[mask, 'Описание'] = ebt_use.loc[mask, 'Описание'] + ' Данные за 2021 и 2023 год — рассчет «Горизонтальной России». Источник тот же.'

# добавляю данные из КОУЖ
ebt_use = ebt_use.merge(
    kouzh_combined,
    on=['Номер региона (ОКАТО)', 'Номер показателя (ID)'],
    how='left',
)
ebt_use[2022] = ebt_use[2022].fillna(ebt_use['2022'])
ebt_use[2024] = ebt_use[2024].fillna(ebt_use['2024'])
ebt_use.drop(columns=['2022', '2024'], inplace=True)
mask = ebt_use['Номер показателя (ID)'] == 95853
ebt_use.loc[mask, 'Описание'] = ebt_use.loc[mask, 'Описание'] + ' Данные за 2022 и 2024 год — рассчет «Горизонтальной России». Источник тот же.'
# ebt_use.head()

/var/folders/5r/7bhf5tqj0wb9bbk62kmr6wk80000gp/T/ipykernel_2691/3995853605.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ebt_use[2021] = ebt_use[2021].fillna(ebt_use['2021'])
/var/folders/5r/7bhf5tqj0wb9bbk62kmr6wk80000gp/T/ipykernel_2691/3995853605.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ebt_use[2023] = ebt_use[2023].fillna(ebt_use['2023'])


In [150]:
# добавляю данные из КОУЖ
kouzh_combined = kouzh_combined[['Номер региона (ОКАТО)', 'Номер показателя (ID)', '2022', '2024']]

ebt_use = ebt_use.merge(
    kouzh_combined,
    on=['Номер региона (ОКАТО)', 'Номер показателя (ID)'],
    how='left',
)

ebt_use[2022] = ebt_use[2022].fillna(ebt_use['2022'])
ebt_use[2024] = ebt_use[2024].fillna(ebt_use['2024'])
ebt_use.drop(columns=['2022', '2024'], inplace=True)

mask = ebt_use['Номер показателя (ID)'] == 95853
ebt_use.loc[mask, 'Описание'] = ebt_use.loc[mask, 'Описание'] + ' Данные за 2022 и 2024 год — рассчет «Горизонтальной России». Источник тот же.'

In [151]:
# ebt_use.to_csv('/Users/Helen/Downloads/ebt_use_step_1.csv', index=False)

### Ожидаемая продолжительность жизни

In [152]:
# ожидаемая продолжительность жизни при рождении, женщины 2014-2026
# она отличается от данных ЕБТ, потому что Росстат задним числом пересчитал показатели
life_exp = pd.read_excel('/Users/Helen/Downloads/Ожидаемая продолжительность населения при рождении, женщины, по регионам, 2014-2023.xls')

life_exp.columns = ['Название региона'] + list(life_exp.iloc[1, 1:])
life_exp.columns = [life_exp.columns[0]] + [int(col) for col in life_exp.columns[1:]]
life_exp['Название региона'] = life_exp['Название региона'].str.strip()
life_exp = life_exp.drop(index=[0, 1])

# добавим номер показателя и название показателя как у ЕБТ
life_exp['Название показателя'] = 'Ожидаемая продолжительность жизни, женщины'
life_exp['Номер показателя (ID)'] = 95741
life_exp['Ед. измерения'] = 'год'
life_exp['Источник'] = 'ЕМИСС'
life_exp['Описание'] = 'Ожидаемая продолжительность жизни женщин при рождении'

life_exp = life_exp[['Название региона', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения', 
                   'Источник',
                   'Описание',
                   2014, 2015, 2016, 2017, 2018,
                   2019, 2020, 2021, 2022, 2023]]

life_exp = vlookup_okato(life_exp, lookup, 'life_exp')
life_exp.drop(life_exp[life_exp['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
# life_exp.sort_values(by='Название региона', ascending=True).head()


In [153]:
ebt_use = pd.concat([ebt_use, life_exp], ignore_index=True)
ebt_use.tail()

# ebt_use = pd.read_csv('/Users/Helen/Downloads/ebt_use_step_1.csv')

# ebt_use.to_csv('/Users/Helen/Downloads/ebt_use_step_2.csv', index=False)

,Номер региона (ОКАТО),Название региона,Название показателя,Номер показателя (ID),Ед. измерения,Источник,Описание,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
427,10000000,Амурская область,"Ожидаемая продолжительность жизни, женщины",95741,год,ЕМИСС,Ожидаемая продолжительность жизни женщин при рождении,72.94,73.11,74.02,74.39,73.90,73.61,72.47,70.81,74.22,74.74,NaN
428,44000000,Магаданская область,"Ожидаемая продолжительность жизни, женщины",95741,год,ЕМИСС,Ожидаемая продолжительность жизни женщин при рождении,72.95,73.06,74.17,75.09,75.20,74.78,74.45,71.92,74.72,75.33,NaN
429,64000000,Сахалинская область,"Ожидаемая продолжительность жизни, женщины",95741,год,ЕМИСС,Ожидаемая продолжительность жизни женщин при рождении,74.07,74.02,74.60,76.12,75.49,75.76,75.09,73.39,75.68,77.30,NaN
430,99000000,Еврейская автономная область,"Ожидаемая продолжительность жизни, женщины",95741,год,ЕМИСС,Ожидаемая продолжительность жизни женщин при рождении,71.23,71.39,72.18,74.16,73.76,72.67,72.45,70.45,73.75,74.38,NaN
431,77000000,Чукотский автономный округ,"Ожидаемая продолжительность жизни, женщины",95741,год,ЕМИСС,Ожидаемая продолжительность жизни женщин при рождении,66.48,69.25,69.23,71.39,67.66,72.39,68.33,68.72,71.31,76.39,NaN


### Доля женщин-потерпевших

In [154]:
# женское население 2014-2025 по регионам
population_zh = pd.read_excel('/Users/Helen/Downloads/Численность постоянного населения - женщин по возрасту на 1 января.xls')
population_zh.columns = ['Название региона'] + list(population_zh.iloc[1, 1:])
population_zh.columns = [population_zh.columns[0]] + [int(col) for col in population_zh.columns[1:]]
population_zh['Название региона'] = population_zh['Название региона'].str.strip()
population_zh = population_zh.drop(index=[0, 1])

population_zh = vlookup_okato(population_zh, lookup, 'population_zh')
population_zh.drop(population_zh[population_zh['Номер региона (ОКАТО)'] == '0'].index, inplace=True)

# population_zh.sort_values(2014, ascending=True).head()

In [155]:
# Количество потерпевших-женщин (человек)
victims = pd.read_excel('/Users/Helen/Downloads/Количество потерпевших-женщин, по регионам, 2014-2025.xls')
victims.columns = ['Название региона'] + list(victims.iloc[1, 1:])
victims.columns = [victims.columns[0]] + [int(col) for col in victims.columns[1:]]
victims = victims.drop(index=[0, 1, 2])

# добавим номер показателя и название показателя как у ЕБТ
victims['Название показателя'] = 'Количество потерпевших-женщин'
victims['Номер показателя (ID)'] = 95912
victims['Ед. измерения'] = 'человек'

# чистим названия Забайкальского края, Крыма, Бурятии и Севастополя, сливаем данные из двух строк в одну 
victims['Название региона'] = victims['Название региона'].str.replace('\\', '', regex=False)
victims.iloc[93, 1:5] = victims.iloc[82, 1:5]
victims.iloc[46, 1:3] = victims.iloc[103, 1:3]
victims.iloc[51, 1:3] = victims.iloc[104, 1:3]
victims.iloc[91, 1:5] = victims.iloc[78, 1:5]

victims = victims.drop(index=[85, 106, 107, 81, 31]) # удаляем ненужные строки, включая ГУ МВД России по г.Санкт-Петербургу и Ленинградской области

victims['Источник'] = 'ЕМИСС по данным МВД'
victims['Описание'] = 'Количество потерпевших - женщин'

victims = victims[['Название региона', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения', 
                   'Источник',
                   'Описание',
                   2014, 2015, 2016, 2017, 2018,
                   2019, 2020, 2021, 2022, 2023,
                   2024, 2025]]

victims = vlookup_okato(victims, lookup, 'victims')
victims.drop(victims[victims['Номер региона (ОКАТО)'] == '0'].index, inplace=True)

In [156]:
# подсчитываю долю женщин-потерпевших от женского населения по регионам
victims_population = victims.merge(
    population_zh, 
    on='Номер региона (ОКАТО)', 
    how='left', 
    suffixes=('_victims', '_population'))

for i in year_cols:
    for j in victims_population['Номер региона (ОКАТО)'].unique():
        if not victims_population.loc[victims_population['Номер региона (ОКАТО)'] == j, f'{i}_population'].eq(0).any():
            victims_population.loc[victims_population['Номер региона (ОКАТО)'] == j, str(i)] = (
                victims_population.loc[victims_population['Номер региона (ОКАТО)'] == j, f'{i}_victims'] /
                victims_population.loc[victims_population['Номер региона (ОКАТО)'] == j, f'{i}_population'] * 100,
                )
    victims_population = victims_population.drop(columns=[f'{i}_victims',f'{i}_population'])

# доля женщин-потерпевших от женского населения
victims_population = victims_population.rename(columns={'Название региона_victims': 'Название региона'})
victims_population = victims_population.drop(columns={'Название региона_population'})

victims_population['Источник'] = 'ЕМИСС по данным МВД'
victims_population['Описание'] = 'Рассчеты «Горизонтальной России». Доля женского населения, потерпевшая от преступлений.'
victims_population['Ед. измерения'] = '%'
victims_population['Номер показателя (ID)'] = 95914
victims_population['Название показателя'] = 'Доля женщин-потерпевших'

victims_population.columns = [i for i in victims_population.columns[:7]] + [int(i) for i in victims_population.columns[7:]]

In [157]:
# ebt_use = pd.read_csv('/Users/Helen/Downloads/ebt_use_step_2.csv')
ebt_use = pd.concat([ebt_use, victims_population], ignore_index=True)
ebt_use.tail()
# ebt_use.to_csv('/Users/Helen/Downloads/ebt_use_step_3.csv')

,Номер региона (ОКАТО),Название региона,Название показателя,Номер показателя (ID),Ед. измерения,Источник,Описание,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
515,10000000,Амурская область,Доля женщин-потерпевших,95914,%,ЕМИСС по данным МВД,"Рассчеты «Горизонтальной России». Доля женского населения, потерпевшая от преступлений.",1.754238,1.82063,1.57507,1.440412,1.435024,1.622816,1.568734,1.492047,1.285279,1.322371,1.288218,1.032952
516,44000000,Магаданская область,Доля женщин-потерпевших,95914,%,ЕМИСС по данным МВД,"Рассчеты «Горизонтальной России». Доля женского населения, потерпевшая от преступлений.",1.081523,1.148364,1.094145,1.017579,1.120139,1.151405,1.149377,0.961198,1.041816,1.054428,0.987042,0.868634
517,64000000,Сахалинская область,Доля женщин-потерпевших,95914,%,ЕМИСС по данным МВД,"Рассчеты «Горизонтальной России». Доля женского населения, потерпевшая от преступлений.",1.677767,1.556524,1.882268,1.448699,1.260275,1.129243,1.046194,1.042766,0.955269,1.08151,1.06469,0.897222
518,99000000,Еврейская автономная область,Доля женщин-потерпевших,95914,%,ЕМИСС по данным МВД,"Рассчеты «Горизонтальной России». Доля женского населения, потерпевшая от преступлений.",1.404147,1.417228,1.260766,1.301584,1.465862,1.400715,1.432616,1.325935,1.32967,1.289226,1.239525,1.071205
519,77000000,Чукотский автономный округ,Доля женщин-потерпевших,95914,%,ЕМИСС по данным МВД,"Рассчеты «Горизонтальной России». Доля женского населения, потерпевшая от преступлений.",0.868084,1.002159,1.145791,0.842219,1.063741,0.965661,1.00434,1.005025,1.104213,1.16688,1.222504,0.981358


In [158]:
# # Численность постоянного населения в среднем за год по регионам, 2014-2024
# population = pd.read_excel('/Users/Helen/Downloads/Численность постоянного населения в среднем за год по регионам, 2014-2024.xls')

# population.iloc[3, 2:12] = population.iloc[2, 2:12]
# population = population.drop(columns='Unnamed: 1')
# population.columns = ['Название региона'] + list(population.iloc[1, 1:])
# population.columns = [population.columns[0]] + [str(col).rstrip('.0') for col in population.columns[1:]]
# population = population.rename(columns={'202': '2020'})
# population = population.drop(index=[0, 1, 2, 28])
# population['Название региона'] = population['Название региона'].str.strip()

### Аборты

In [159]:
# какие регионы запретили склонение к абортам
abortion_ban = pd.read_csv('/Users/Helen/Downloads/запрет абортов_их пропаганды.csv')
abortion_ban.drop(columns=['Запрет на проведение абортов', 'проведение ссылка', 'доля абортов (Патриар'], inplace=True)
abortion_ban = abortion_ban.rename(columns={'Регион': 'Название региона',
                                            'пропаганда ссылка': 'Описание', 
                                            'Запрет на пропаганду абортов': 2026})

# добавим номер показателя и название показателя как у ЕБТ
abortion_ban['Название показателя'] = 'Запрет на пропаганду абортов'
abortion_ban['Номер показателя (ID)'] = 95911
abortion_ban['Ед. измерения'] = 'оценка'
abortion_ban['Источник'] = 'Заксобрание региона'
abortion_ban['Описание'] = 'Принят ли в регионе «О запрете склонения женщин к искусственному прерыванию беременности».'

abortion_ban = abortion_ban[['Название региона', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения', 
                   'Источник',
                   'Описание',
                   2026]]

abortion_ban = vlookup_okato(abortion_ban, lookup, 'abortion_ban')
abortion_ban.drop(abortion_ban[abortion_ban['Номер региона (ОКАТО)'] == '0'].index, inplace=True)

In [160]:
# добавляю данные о запрете пропаганды абортов в общий датасет
ebt_use = pd.concat([ebt_use, abortion_ban], ignore_index=True)

# присваиваем значения 0 и 1 вместо "да" и "нет" для показателя "Запрет на пропаганду абортов"
# значения бинарных переменных перевернуты, так как запрет на «пропаганду абортов» — ограничение прав женщин 
ebt_use.loc[ebt_use['Название показателя'] == 'Запрет на пропаганду абортов', 2026] = \
    ebt_use.loc[ebt_use['Название показателя'] == 'Запрет на пропаганду абортов', 2026].map({'да': 0, 'нет': 1})
ebt_use.tail()

,Номер региона (ОКАТО),Название региона,Название показателя,Номер показателя (ID),Ед. измерения,Источник,Описание,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
604,96000000,Чечня,Запрет на пропаганду абортов,95911,оценка,Заксобрание региона,Принят ли в регионе «О запрете склонения женщин к искусственному прерыванию беременности».,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
605,97000000,Чувашия,Запрет на пропаганду абортов,95911,оценка,Заксобрание региона,Принят ли в регионе «О запрете склонения женщин к искусственному прерыванию беременности».,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
606,77000000,Чукотский АО,Запрет на пропаганду абортов,95911,оценка,Заксобрание региона,Принят ли в регионе «О запрете склонения женщин к искусственному прерыванию беременности».,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
607,71140000,Ямало-Ненецкий АО,Запрет на пропаганду абортов,95911,оценка,Заксобрание региона,Принят ли в регионе «О запрете склонения женщин к искусственному прерыванию беременности».,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
608,78000000,Ярославская обл.,Запрет на пропаганду абортов,95911,оценка,Заксобрание региона,Принят ли в регионе «О запрете склонения женщин к искусственному прерыванию беременности».,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


### Данные ЕБТ по абортам

In [161]:
# загружаю данные об абортах, оставляю нужное
# количество нелегальных абортов на 1000 женщин по регионам и годам, 2014-2024
df = pd.read_csv(
	'../data/data_abortions_119_v20251007_csv/data_abortions_119_v20251007_wide.csv',
	sep=';',
    engine='python',
	on_bad_lines='skip',
	encoding='utf-8'
)

# drop columns year_2000 to year_2014
df = df.drop(columns=[col for col in df.columns 
                      if col.startswith('year_') and 
                      int(col.split('_')[1]) >= 2000 and 
                      int(col.split('_')[1]) < 2014])

# df = df[df['object_level'] == 'регион']
df = df.dropna(subset=df.columns[12:23], how='all')

In [162]:
# выделяю данные об общем количестве абортов на 1000 женщин
# pd.set_option('display.max_columns', None)
abortion = df[(df['object_level'].isin(['страна', 'регион'])) &
            (df['group'] == 'Всего') & 
            (df['indicator_code'] == 'Y477000008') & 
            (df['organization_type'] == 'Всего') & 
            (df['age'] == 'Всего') & 
            (df['pregnancy_duration'] == 'Всего') &
            (df['indicator_unit'] == 'На 1000 женщин соответствующего возраста')]

abortion.columns = [col if not col.startswith('year_') 
                    else int(col.split('_')[1]) for col in abortion.columns]

# привожу данные об абортах к виду основного датасета
abortion = abortion.drop(columns=['object_level', 
                                  'object_oktmo',
                                  'source'])
abortion = abortion.rename(columns={'object_name': 'Название региона',
                                            'object_okato': 'Номер региона (ОКАТО)', 
                                            'comment': 'Описание'})
# присваиваю РФ искуственный номер ОКАТО
ru = abortion[abortion['Название региона'] == 'Российская Федерация'].index
abortion.loc[ru, 'Номер региона (ОКАТО)'] = '100000000000'

abortion['Название показателя'] = 'Всего абортов на 1000 женщин'
abortion['Номер показателя (ID)'] = 'Y477000008'
abortion['Ед. измерения'] = 'человек'
abortion['Описание'] = ('Расчет «Если быть точным» по данным Росстата.' 
                        'Номер показателя (ID) заимствован у «Если быть точным» из датасета «Аборты в России»')
abortion['Источник'] = 'Росстат'

abortion = abortion[['Номер региона (ОКАТО)',
                            'Название региона', 
                            'Название показателя',
                            'Номер показателя (ID)', 
                            'Ед. измерения', 
                            'Источник',
                            'Описание',
                            2014, 2015, 2016, 2017, 2018,
                            2019, 2020, 2021, 2022, 2023,
                            2024]]

/var/folders/5r/7bhf5tqj0wb9bbk62kmr6wk80000gp/T/ipykernel_2691/405117630.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '100000000000' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  abortion.loc[ru, 'Номер региона (ОКАТО)'] = '100000000000'


In [163]:
# данные о криминальных абортах в общий датасет
# выделяю данные о количестве криминальных абортов на 1000 женщин по регионам и годам
illegal = df[(df['object_level'].isin(['страна', 'регион'])) &
         (df['group'] == 'Всего') & 
         (df['indicator_code'] == 'Y477000010') & 
         (df['organization_type'] == 'Всего') & 
         (df['age'] == 'Всего') & 
         (df['indicator_unit'] == 'На 1000 женщин соответствующего возраста')]

illegal.columns = [col if not col.startswith('year_') 
                    else int(col.split('_')[1]) for col in illegal.columns]

illegal = illegal.rename(columns={'object_name': 'Название региона',
                                    'object_okato': 'Номер региона (ОКАТО)', 
                                    'comment': 'Описание'})

# удаляю неполные данные Минздрава о общем количестве крим. абортов на 1000 женщин в России
i = illegal[(illegal['object_level'] == 'страна') &
         (illegal['pregnancy_duration'] == 'Всего')].index
illegal.drop(index=i, inplace=True)
ru = illegal[illegal['Название региона'] == 'Российская Федерация'].index
illegal.loc[ru, 'Номер региона (ОКАТО)'] = '100000000000'

# добавляю разные ID показателя в зависимости от длительности беременности
illegal['Название показателя'] = illegal['pregnancy_duration'].map({
                                          'До 12 недель': 'Криминальные аборты на 1000 женщин, до 12 недель (всего)', 
                                          '12-21 недель': 'Криминальные аборты на 1000 женщин, 12-21 неделя (всего)'
                                          })
# добавляю разные ID показателя в зависимости от длительности беременности
illegal['Номер показателя (ID)'] = illegal['pregnancy_duration'].map({
                                                'До 12 недель': 'Y477000010.11', 
                                                '12-21 недель': 'Y477000010.13'
                                                })

illegal['Ед. измерения'] = 'человек'
illegal['Описание'] = ('Расчет «Если быть точным» по данным Росстата. '
                        'Номер показателя (ID) Y477000010 заимствован у «Если быть точным» из датасета «Аборты в России», '
                        'к нему добавлены суффиксы в зависимости от длительности беременности')
illegal['Источник'] = 'Росстат'

illegal = illegal.drop(columns=['object_level', 
                                  'object_oktmo',
                                  'source', 
                                  'pregnancy_duration'])

illegal = illegal[['Номер региона (ОКАТО)',
                     'Название региона', 
                     'Название показателя',
                     'Номер показателя (ID)', 
                     'Ед. измерения', 
                     'Источник',
                     'Описание',
                  #  2014, 2015, 2016, 2017, 
                     2018, 2019, 2020, 2021, 
                     2022, 2023, 2024]]

/var/folders/5r/7bhf5tqj0wb9bbk62kmr6wk80000gp/T/ipykernel_2691/2166771615.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '100000000000' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  illegal.loc[ru, 'Номер региона (ОКАТО)'] = '100000000000'


In [164]:
# добавляю данные об абортах на 1000 женщин в общий датасет
ebt_use = pd.concat([ebt_use, abortion, illegal], ignore_index=True)
ebt_use.tail()

,Номер региона (ОКАТО),Название региона,Название показателя,Номер показателя (ID),Ед. измерения,Источник,Описание,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
868,96000000,Чеченская Республика,"Криминальные аборты на 1000 женщин, до 12 недель (всего)",Y477000010.11,человек,Росстат,Расчет «Если быть точным» по данным Росстата. Номер показателя (ID) Y477000010 заимствован у «Ес...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
869,97000000,Чувашская Республика,"Криминальные аборты на 1000 женщин, до 12 недель (всего)",Y477000010.11,человек,Росстат,Расчет «Если быть точным» по данным Росстата. Номер показателя (ID) Y477000010 заимствован у «Ес...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
870,77000000,Чукотский автономный округ,"Криминальные аборты на 1000 женщин, до 12 недель (всего)",Y477000010.11,человек,Росстат,Расчет «Если быть точным» по данным Росстата. Номер показателя (ID) Y477000010 заимствован у «Ес...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
871,71140000,Ямало-Ненецкий автономный округ,"Криминальные аборты на 1000 женщин, до 12 недель (всего)",Y477000010.11,человек,Росстат,Расчет «Если быть точным» по данным Росстата. Номер показателя (ID) Y477000010 заимствован у «Ес...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
872,78000000,Ярославская область,"Криминальные аборты на 1000 женщин, до 12 недель (всего)",Y477000010.11,человек,Росстат,Расчет «Если быть точным» по данным Росстата. Номер показателя (ID) Y477000010 заимствован у «Ес...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


In [165]:
ebt_use.to_csv('/Users/Helen/Downloads/ebt_use_step_4.csv')

In [166]:
# # проверяю, за какие годы нет данных по абортам
# pd.set_option('display.max_rows', None)
# df.groupby('object_name').apply(lambda x: x.filter(regex='^year_202').isna().sum())

# # доля пропущенных значений от всех значений данного года в данном регионе
# share_of_missing = df.groupby('object_name').apply(lambda x: x.filter(regex='^year_202').isna().sum() / (x.filter(regex='^year_202').count() + x.filter(regex='^year_202').isna().sum()))
# share_of_missing.sort_values('year_2024',ascending=False)

## Доступность продуктов

In [167]:
# стоимость продуктовой корзины в октябре (во время опроса о средних зп) с 2014 по 2026 гг.
groceries = pd.read_excel('/Users/Helen/Downloads/Стоимость фиксированного набора потребительских товаров и услуг по регионам, 2014-2026.xls')

groceries.columns = ['Название региона'] + list(groceries.iloc[1, 1:])
groceries.columns = [groceries.columns[0]] + [int(col) for col in groceries.columns[1:]]
groceries['Название региона'] = groceries['Название региона'].str.strip()
groceries.iloc[3, 10:] = groceries.iloc[4, 10:]
groceries = groceries.drop(index=[0, 1, 2, 4, 29, 77]) # удаляю ненужные или дублирующиеся строки, пр., "Архангельская область (без АО)" и "Тюменская область (без АО)"

groceries = vlookup_okato(groceries, lookup, 'groceries')
groceries.drop(groceries[groceries['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
print(f"groceries has {groceries['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")

groceries has 0 missing values in 'Номер региона (ОКАТО)' column.


In [168]:
# доля корзины от средней зарплаты женщин
salary_long_w = ebt_use[ebt_use['Номер показателя (ID)'] == 95790] # выделяю только среднюю зарплату женщин, которую буду использовать в подсчетах
year_columns = [2015, 2017, 2019, 2021, 2023]

# salary_long_w.loc[:, 'Номер региона (ОКАТО)'] = salary_long_w.loc[:, 'Номер региона (ОКАТО)'].str.replace('.0', '', regex=False)
for i in year_columns:
    salary_long_w.loc[:, i] = pd.to_numeric(salary_long_w.loc[:, i], errors='coerce')

# соединяю два датасета
affordability = salary_long_w[['Название региона', 'Номер региона (ОКАТО)', 2015, 2017, 2019, 2021, 2023]].merge(
    groceries[['Номер региона (ОКАТО)', 2015, 2017, 2019, 2021, 2023]],
    on=['Номер региона (ОКАТО)'],
    suffixes=('_wages', '_groceries'),
    how='left'
)

affordability['Номер показателя (ID)'] = 95913
affordability['Название показателя'] = 'Доля расходов на питание'
affordability['Ед. измерения'] = '%'
affordability['Описание'] = 'Расчет «Горизонтальной России». Средняя  начисленная зарплата мужчин и женщин по группам занятий' \
                         'и субъектам Российской Федерации за октябрь отчетного года.' \
                         'Источник - Росстат: Сведения о заработной плате работников в организациях по категориям персонала и профессиональным группам работников.' \
                         '\nСтоимость продуктовой корзины. Источник - Росстат: Стоимость фиксированного набора потребительских товаров и услуг.'
affordability['Источник'] = 'Источник данных о зарплате женщин — Росстат, о стоимости продуктовой корзины — ЕМИСС, Росстат.'

# добавляю рассчет доли корзины от средней зарплаты женщин в процентах 2014-2023
for i in year_columns:
    affordability[i] = affordability[f'{i}_groceries'] / affordability[f'{i}_wages'] * 100

affordability = affordability.drop(columns=[f'{i}_wages' for i in year_columns] + [f'{i}_groceries' for i in year_columns])
affordability.sort_values(by='Название региона', ascending=True).head()
print(f"affordability has {affordability['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")


affordability has 0 missing values in 'Номер региона (ОКАТО)' column.


## Реальная зарплата

In [169]:
# индексе потребительских цен от года к году по регионам, 2014-2018
cpi = pd.read_excel('/Users/Helen/Downloads/Индексы потребительских цен на товары и услуги, 2014-2024.xls')
cpi.columns = ['Название региона'] + list(cpi.iloc[1, 1:])
cpi = cpi.drop(columns=cpi.columns[1])
cpi.columns = [cpi.columns[0]] + [int(col) for col in cpi.columns[1:]]
cpi['Название региона'] = cpi['Название региона'].str.strip()
cpi.iloc[4, -2:] = cpi.iloc[5, -2:]
cpi = cpi.drop(index=[0, 1, 2, 3, 5])

cpi = vlookup_okato(cpi, lookup, 'cpi')
cpi.drop(cpi[cpi['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
print(f"cpi has {cpi['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")
# cpi

cpi has 0 missing values in 'Номер региона (ОКАТО)' column.


In [170]:
# считаю реальную зп женщин с помощью индекса потребительских цен
# соединяю два датасета
year_columns = [2015, 2017, 2019, 2021, 2023]

real_wages = salary_long_w[['Название региона', 'Номер региона (ОКАТО)', 2015, 2017, 2019, 2021, 2023]].merge(
    cpi[['Номер региона (ОКАТО)', 2015, 2017, 2019, 2021, 2023]],
    on=['Номер региона (ОКАТО)'],
    suffixes=('_wages', '_cpi'),
    how='left'
)

real_wages['Номер показателя (ID)'] = 95915
real_wages['Название показателя'] = 'Реальная средняя начисленная зарплата женщин'
real_wages['Ед. измерения'] = 'руб.'
real_wages['Описание'] = 'Расчет «Горизонтальной России». Средняя  начисленная зарплата мужчин и женщин по группам занятий' \
                         'и субъектам Российской Федерации за октябрь отчетного года.' \
                         'Источник - Росстат: Сведения о заработной плате работников в организациях по категориям персонала и профессиональным группам работников.' \
                         '\nИндексы потребительских цен на товары и услуги к соответствующему периоду предыдущего года, данные за каждый октябрь.' \
                         'Оккупированные территории ДНР, ЛНР, Запорожской и Херсонской областей не включены в рассчеты. Источник - Росстат.'
real_wages['Источник'] = 'Источник данных о зарплате женщин и об индексе потребительских цен — Росстат.'

for i in year_columns:
    real_wages.loc[:, i] = real_wages.loc[:, f'{i}_wages'] / (real_wages.loc[:, f'{i}_cpi'] / 100)
    drop_cols = [f'{i}_wages', f'{i}_cpi']
    real_wages = real_wages.drop(columns=drop_cols)

for j in year_columns:
    real_wages[j] = pd.to_numeric(real_wages[j], errors='coerce').round(0)

# real_wages.sort_values(by='Название региона', ascending=True).head()
print(f"real_wages has {real_wages['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")

real_wages has 0 missing values in 'Номер региона (ОКАТО)' column.


In [171]:
ebt_use = pd.concat([ebt_use, affordability, real_wages], ignore_index=True)
# ebt_use['Название показателя'].unique()

In [172]:
def okato_to_region(df, lookup_df, region_col):
    lookup_df = lookup_df[['object_okato', region_col]].copy()
    lookup_df[region_col] = lookup_df[region_col].astype(str)
    lookup_df['object_okato'] = lookup_df['object_okato'].astype(str).str.replace('.0', '', regex=False).str.strip()

    lookup_df = lookup_df.sort_values(by=region_col, na_position='last').drop_duplicates('object_okato')

    df = df.copy()
    df['Номер региона (ОКАТО)'] = df['Номер региона (ОКАТО)'].astype(str).str.replace('.0', '', regex=False).str.strip()

    mapping = dict(zip(lookup_df['object_okato'], lookup_df[region_col]))
    df.insert(loc=0, column='datawrapper', value=df['Номер региона (ОКАТО)'].map(mapping))
    return df

ebt_use = okato_to_region(ebt_use, lookup, 'datawrapper')
ebt_use = ebt_use.drop(columns='Название региона')
ebt_use = ebt_use.rename(columns={'datawrapper':'Название региона'})

# missing = sorted(set(ebt_use.loc[ebt_use['datawrapper'].isna(), 'Номер региона (ОКАТО)']))
# print('Missing OKATO codes:', missing)
ebt_use.head()

,Название региона,Номер региона (ОКАТО),Название показателя,Номер показателя (ID),Ед. измерения,Источник,Описание,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
0,Российская Федерация,100000000000,"Средняя заработная плата по группам занятий, мужчины",95789,рубль,NaN,Средняя начисленная заработная плата мужчин и женщин по группам занятий и субъектам Российской ...,NaN,42573.0,NaN,50842.0,NaN,59656.0,NaN,71364.0,NaN,96425.0,NaN,NaN,NaN
1,Российская Федерация,100000000000,"Средняя заработная плата по группам занятий, женщины",95790,рубль,NaN,Средняя начисленная заработная плата мужчин и женщин по группам занятий и субъектам Российской ...,NaN,29235.0,NaN,34480.0,NaN,41233.0,NaN,49205.0,NaN,64591.0,NaN,NaN,NaN
2,Российская Федерация,100000000000,Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин),95799,%,NaN,Расчет «Если быть точным». Гендерный разрыв рассчитывается по методологии МОТ как отношение разн...,NaN,31.3,NaN,32.2,NaN,30.9,NaN,31.05,NaN,33.01,NaN,NaN,NaN
3,Российская Федерация,100000000000,Забота о других,95853,%,NaN,"Расчет «Если быть точным». Доля респондентов, в круг ежедневных занятий которых входит уход за д...",NaN,NaN,24.0,NaN,31.0,NaN,47.0,NaN,39.0,NaN,36.0,NaN,NaN
4,Алтайский край,1000000,"Средняя заработная плата по группам занятий, женщины",95790,рубль,NaN,Средняя начисленная заработная плата мужчин и женщин по группам занятий и субъектам Российской ...,NaN,17985.0,NaN,19968.0,NaN,26928.0,NaN,30064.0,NaN,41628.0,NaN,NaN,NaN


In [173]:
# ebt_use = pd.read_csv('/Users/Helen/Downloads/ebt_use_step_5.csv')
# ebt_use = ebt_use.drop(columns=['Unnamed: 0'])
ebt_use['Номер региона (ОКАТО)'] = ebt_use['Номер региона (ОКАТО)'].astype(int)

# заполняю пропуски данными прошлого года для показателя "Забота о других" (ID 95853)
mask = ebt_use['Номер показателя (ID)'] == 95853
ebt_use.loc[mask, [y for y in year_cols]] = ebt_use.loc[mask, [y for y in year_cols]].ffill(axis=1)
ebt_use.loc[mask, [y for y in year_cols]]

/var/folders/5r/7bhf5tqj0wb9bbk62kmr6wk80000gp/T/ipykernel_2691/822201602.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ebt_use.loc[mask, [y for y in year_cols]] = ebt_use.loc[mask, [y for y in year_cols]].ffill(axis=1)


,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
3,NaN,NaN,24.0,24.0,31.0,31.0,47.0,47.0,39.0,39.0,36.0,36.0
6,NaN,NaN,25.0,25.0,31.0,31.0,46.0,46.0,34.0,34.0,30.0,30.0
9,NaN,NaN,27.0,27.0,29.0,29.0,52.0,52.0,43.0,43.0,41.0,41.0
12,NaN,NaN,23.0,23.0,33.0,33.0,54.0,54.0,42.0,42.0,35.0,35.0
17,NaN,NaN,24.0,24.0,30.0,30.0,39.0,39.0,41.0,41.0,32.0,32.0
...,...,...,...,...,...,...,...,...,...,...,...,...
325,NaN,NaN,22.0,22.0,27.0,27.0,53.0,53.0,36.0,36.0,45.0,45.0
330,NaN,NaN,24.0,24.0,30.0,30.0,50.0,50.0,39.0,39.0,37.0,37.0
334,NaN,NaN,24.0,24.0,43.0,43.0,45.0,45.0,36.0,36.0,42.0,42.0
339,NaN,NaN,27.0,27.0,26.0,26.0,48.0,48.0,41.0,41.0,25.0,25.0


In [174]:
mask = ebt_use['Номер показателя (ID)'] == '95853'
subset = ebt_use.loc[mask]

print(subset.isna().sum())
print('total NaNs:', subset.isna().sum().sum())

Название региона         0
Номер региона (ОКАТО)    0
Название показателя      0
Номер показателя (ID)    0
Ед. измерения            0
Источник                 0
Описание                 0
2014                     0
2015                     0
2016                     0
2017                     0
2018                     0
2019                     0
2020                     0
2021                     0
2022                     0
2023                     0
2024                     0
2025                     0
2026                     0
dtype: int64
total NaNs: 0


In [175]:
ebt_use.to_csv('/Users/Helen/Downloads/ebt_use_step_5.csv')

In [176]:
# missing = sorted(set(ebt_use.loc[ebt_use['Название региона'].isna(), 'Номер региона (ОКАТО)']))
# print('Missing OKATO codes:', missing)

# # how many values are missing in each Номер показателя (ID) by year 
# missing_by_indicator = ebt_use.groupby('Название показателя')[year_cols].apply(lambda x: x.isna().sum())
# print(missing_by_indicator)

## Корректируем экономические показатели для Северо-Кавказского фед. округа

Как и журналисты из «Если быть точным», я не учитываю данные о зарплатах в регионах Северо-Кавказского федерального округа из-за высокой доли неформального сектора и ненадежности стат. данных. Вместо этого присваиваю всем регионам среднее значение по России

In [177]:
# ebt_use = pd.read_csv('/Users/Helen/Downloads/ebt_use_step_5.csv')
# north_caucasus = ['Республика Дагестан', 'Республика Ингушетия', 'Кабардино-Балкарская Республика',
#                    'Республика Северная Осетия-Алания', 'Чеченская Республика', 
#                    'Карачаево-Черкесская Республика', 'Ставропольский край']
north_okato = [82000000, 26000000, 83000000, 
               90000000, 96000000, 
               91000000, 7000000]

# заменяем значения для средней зарплаты женщин и мужчин, 
# гендерного разрыва в зарплатах, доли расходов на питание и реальной средней зарплаты женщин
ids = ['95789', '95790', '95799', '95913', '95915']

for i in ids:
    for j in north_okato:
        ebt_use.loc[(ebt_use['Номер региона (ОКАТО)'] == j) & 
                    (ebt_use['Номер показателя (ID)'] == i), '2014':'2026'] = ebt_use.loc[(ebt_use['Номер региона (ОКАТО)'] == 100000000000) & 
                                                                                          (ebt_use['Номер показателя (ID)'] == i), '2014':'2026'].values
# ebt_use.loc[(ebt_use['Номер региона (ОКАТО)'].isin(north_okato)) & (ebt_use['Номер показателя (ID)'].isin(ids))]

KeyError: '2014'

# Считаем рейтинг 7х7

In [ ]:
indicators = ebt_use[['Название показателя', 'Номер показателя (ID)']].drop_duplicates()
print(indicators)

                                                      Название показателя  \
0                    Средняя заработная плата по группам занятий, мужчины   
1                    Средняя заработная плата по группам занятий, женщины   
2    Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)   
3                                                         Забота о других   
344                            Ожидаемая продолжительность жизни, женщины   
432                                               Доля женщин-потерпевших   
520                                          Запрет на пропаганду абортов   
609                                          Всего абортов на 1000 женщин   
697              Криминальные аборты на 1000 женщин, 12-21 неделя (всего)   
785              Криминальные аборты на 1000 женщин, до 12 недель (всего)   
873                                              Доля расходов на питание   
959                          Реальная средняя начисленная зарплата женщин   

In [ ]:
# нормализую данные
# ebt_use = pd.read_csv('/Users/Helen/Downloads/ebt_use_step_5.csv')

# показатели, где меньше — лучше, 
# а именно: Доля женщин-потерпевших ('95914'), Время на уход за детьми ('95853'), 
# Криминальные аборты до и после 12 недель ('Y477000010.11', 'Y477000010.13'), 
# гендерный разрыв в зарплатах ('95799') и доля расходов на питание ('95913')
invert_ids = ['95914', '95853', 'Y477000010.11', 'Y477000010.13', '95799', '95913']

# удаляю из датасета показатели, не учитывающиеся в рейтинге: 
# Запрет на пропаганду абортов ('95911'), кол-во абортов на 1000 женщин ('Y477000008'),
# включаю среднюю зарплату мужчин ('95789') для корректирующего показателя
ebt_use_2 = ebt_use[~ebt_use['Номер показателя (ID)'].isin(['95911', 'Y477000008'])]
odd_years = [str(year) for year in range(2015, 2026, 2)] 

def normalize_variable(df, invert):
    result = df.copy()
    for year in odd_years:
        col = pd.to_numeric(df[year], errors='coerce')
        x_min, x_max = col.min(), col.max()
        normalized = (col - x_min) / (x_max - x_min)
        result[year] = 1 - normalized if invert else normalized
    return result

normalized_parts = []
for var_id, group in ebt_use_2.groupby('Номер показателя (ID)'):
    invert = var_id in invert_ids
    normalized_parts.append(normalize_variable(group, invert=invert))

ebt_normalized = pd.concat(normalized_parts).sort_index()

# корректирую показатель о гендерном разрыве зарплат, учитывая уровень зарплат мужчин
# тогда регионы с небольшим разрывом между мужчинами и женщинами при низких зарплатах не окажутся лидерами рейтинга 
var_95799 = ebt_normalized[ebt_normalized['Номер показателя (ID)'] == '95799'].set_index('Номер региона (ОКАТО)')
var_95789 = ebt_normalized[ebt_normalized['Номер показателя (ID)'] == '95789'].set_index('Номер региона (ОКАТО)')

composite_rows = var_95799.copy()
composite_rows['Номер показателя (ID)'] = '95917'
for year in odd_years:
    composite_rows[year] = var_95799[year] * var_95789[year]
composite_rows = composite_rows.reset_index()

ebt_normalized = pd.concat([ebt_normalized, composite_rows]).sort_index()
ebt_normalized = ebt_normalized[~ebt_normalized['Номер показателя (ID)'].isin(['95789', '95790'])]

In [ ]:
for year in odd_years:
    means = ebt_normalized.groupby('Номер региона (ОКАТО)')[year].mean()
    ebt_normalized[f'Рейтинг 7х7_{year}'] = ebt_normalized['Номер региона (ОКАТО)'].map(means)

# ebt_normalized.to_csv('/Users/Helen/Downloads/ebt_normalized.csv', index=False)

# assign categorical value to every region on every odd year by quantiles of Рейтинг 7х7 columns
rating_cols = [col for col in ebt_normalized.columns if col.startswith('Рейтинг 7х7')]
for col in rating_cols:
    ebt_normalized[f'{col}_оценка'] = pd.cut(ebt_normalized[col], bins=5, labels=['Кол', 'Неуд', 'Удовлетворительно', 'Хорошо', 'Отлично'])
ebt_normalized.head()

,datawrapper,Unnamed: 0,Название региона,Номер региона (ОКАТО),Название показателя,Номер показателя (ID),Ед. измерения,Источник,Описание,2014,...,Рейтинг 7х7_2019,Рейтинг 7х7_2021,Рейтинг 7х7_2023,Рейтинг 7х7_2025,Рейтинг 7х7_2015_оценка,Рейтинг 7х7_2017_оценка,Рейтинг 7х7_2019_оценка,Рейтинг 7х7_2021_оценка,Рейтинг 7х7_2023_оценка,Рейтинг 7х7_2025_оценка
0,Российская Федерация,2,Российская Федерация,100000000000,Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин),95917,%,NaN,Расчет «Если быть точным». Гендерный разрыв рассчитывается по методологии МОТ как отношение разн...,NaN,...,0.510382,0.596373,0.569615,0.525330,Удовлетворительно,Удовлетворительно,Удовлетворительно,Удовлетворительно,Удовлетворительно,Удовлетворительно
1,Алтайский край,7,Алтайский край,1000000,Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин),95917,%,NaN,Расчет «Если быть точным». Гендерный разрыв рассчитывается по методологии МОТ как отношение разн...,NaN,...,0.463098,0.498724,0.494147,0.536649,Неуд,Неуд,Удовлетворительно,Неуд,Неуд,Удовлетворительно
2,Российская Федерация,2,Российская Федерация,100000000000,Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин),95799,%,NaN,Расчет «Если быть точным». Гендерный разрыв рассчитывается по методологии МОТ как отношение разн...,NaN,...,0.510382,0.596373,0.569615,0.525330,Удовлетворительно,Удовлетворительно,Удовлетворительно,Удовлетворительно,Удовлетворительно,Удовлетворительно
2,Амурская область,8,Амурская область,10000000,Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин),95917,%,NaN,Расчет «Если быть точным». Гендерный разрыв рассчитывается по методологии МОТ как отношение разн...,NaN,...,0.398988,0.493025,0.429929,0.280248,Неуд,Кол,Неуд,Неуд,Неуд,Неуд
3,Российская Федерация,3,Российская Федерация,100000000000,Забота о других,95853,%,NaN,"Расчет «Если быть точным». Доля респондентов, в круг ежедневных занятий которых входит уход за д...",NaN,...,0.510382,0.596373,0.569615,0.525330,Удовлетворительно,Удовлетворительно,Удовлетворительно,Удовлетворительно,Удовлетворительно,Удовлетворительно


In [ ]:
ebt_normalized.to_csv('/Users/Helen/Downloads/ebt_ratings.csv', index=False)